# Gold · Dimensión de Productos

## Propósito

Construir la dimensión de productos en la capa Gold con historial de cambios (SCD2), conservando la evolución de precios y clasificaciones a lo largo del tiempo.

## Flujo

1. Leer la versión actual de productos desde la capa Silver.
2. En la primera carga, crear la tabla con todos los productos como versión vigente.
3. En las siguientes ejecuciones, comparar cada producto contra su versión vigente.
4. Cerrar las versiones anteriores de los productos que cambiaron.
5. Insertar las nuevas versiones de los productos nuevos y de los que cambiaron.

## Destino

Capa Gold.

**Tabla:** `gld_dataco.supply_chain.desc_dim_products`

**Llave sustituta:** `producto_key`

## Atributos que generan una nueva versión

* Categoría y nombre de categoría
* Departamento y nombre de departamento
* Nombre y descripción
* Precio y estado

## Control de vigencia

* `es_vigente` indica la versión actual de cada producto.
* Las consultas de negocio filtran `es_vigente = true`.

## Resultado esperado

La dimensión de productos actualizada, con el historial de precios conservado y una única versión vigente por producto.

In [0]:
# ============================================================
# DIM_PRODUCTO — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("silver_catalog", "slv_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "supply_chain", "Esquema Silver")

dbutils.widgets.text("gold_catalog", "gld_dataco", "Catálogo Gold")
dbutils.widgets.text("gold_schema", "supply_chain", "Esquema Gold")

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_catalog = dbutils.widgets.get("gold_catalog")
gold_schema = dbutils.widgets.get("gold_schema")

SILVER_PRODUCTOS_FQN = f"{silver_catalog}.{silver_schema}.md_productos"
GOLD_DIM_PRODUCTO_FQN = f"{gold_catalog}.{gold_schema}.desc_dim_products"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_catalog}.{gold_schema}")

print(f"✓ Origen: {SILVER_PRODUCTOS_FQN}")
print(f"✓ Destino: {GOLD_DIM_PRODUCTO_FQN}")

In [0]:
# ============================================================
# DIM_PRODUCTO — Celda 2: INICIO DE LOG DE EJECUCIÓN — gld_dim_producto
# ============================================================

import uuid

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="id_ejecucion",
    debugValue=""
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "gld_dim_products"
CAPA = "gold"

spark.sql(f"""
INSERT INTO metadata_dataco.supply_chain.hd_etl_log
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
# ============================================================
# DIM_PRODUCTO — Celda 3: Leer versión actual de Silver
# ============================================================

from pyspark.sql import functions as F

df_productos_actual = spark.table(SILVER_PRODUCTOS_FQN).select(
    "product_card_id",
    "product_category_id",
    "category_name",
    "department_id",
    "department_name",
    "product_name",
    "product_description",
    "product_price",
    "product_status"
)

print(f"✓ Productos en Silver: {df_productos_actual.count()}")

In [0]:
# ============================================================
# DIM_PRODUCTO — Celda 4: Crear tabla si no existe (primera carga)
# ============================================================

from delta.tables import DeltaTable

if not spark.catalog.tableExists(GOLD_DIM_PRODUCTO_FQN):

    df_inicial = (
        df_productos_actual
        .withColumn("producto_key", F.monotonically_increasing_id())
        .withColumn("fecha_inicio_vigencia", F.current_date())
        .withColumn("fecha_fin_vigencia", F.lit(None).cast("date"))
        .withColumn("es_vigente", F.lit(True))
    )

    (
        df_inicial.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD_DIM_PRODUCTO_FQN)
    )

    print(f"✓ Tabla {GOLD_DIM_PRODUCTO_FQN} creada con {df_inicial.count()} productos (carga inicial)")
    CARGA_INICIAL = True
else:
    print(f"✓ Tabla {GOLD_DIM_PRODUCTO_FQN} ya existe — se procesa como SCD2")
    CARGA_INICIAL = False

In [0]:
# ============================================================
# DIM_PRODUCTO — Celda 5: Lógica SCD2 (solo si la tabla ya existía)
# ============================================================

if not CARGA_INICIAL:

    delta_dim_producto = DeltaTable.forName(spark, GOLD_DIM_PRODUCTO_FQN)

    df_vigente = (
        delta_dim_producto.toDF()
        .filter(F.col("es_vigente") == True)
    )

    # Columnas que, si cambian, generan una nueva versión del producto
    # (product_price es la más probable en la práctica: cambios de precio)
    COLUMNAS_COMPARADAS = [
        "product_category_id", "category_name", "department_id",
        "department_name", "product_name", "product_description",
        "product_price", "product_status"
    ]

    condicion_cambio = " OR ".join([
        f"v.{c} <> n.{c}" for c in COLUMNAS_COMPARADAS
    ])

    df_comparado = (
        df_productos_actual.alias("n")
        .join(df_vigente.alias("v"), on="product_card_id", how="left")
        .selectExpr(
            "n.*",
            "v.producto_key AS producto_key_existente",
            f"CASE WHEN v.product_card_id IS NULL THEN 'NUEVO' "
            f"WHEN {condicion_cambio} THEN 'CAMBIO' "
            f"ELSE 'SIN_CAMBIO' END AS tipo_evento"
        )
    )

    df_a_cerrar = df_comparado.filter(F.col("tipo_evento") == "CAMBIO")
    df_a_insertar = df_comparado.filter(F.col("tipo_evento").isin("NUEVO", "CAMBIO"))

    print(f"✓ Productos nuevos: {df_comparado.filter(F.col('tipo_evento') == 'NUEVO').count()}")
    print(f"✓ Productos con cambios (nueva versión): {df_a_cerrar.count()}")
    print(f"✓ Productos sin cambios: {df_comparado.filter(F.col('tipo_evento') == 'SIN_CAMBIO').count()}")

In [0]:
# ============================================================
# DIM_PRODUCTO — Celda 6: Cerrar versiones viejas (solo los que cambiaron)
# ============================================================

if not CARGA_INICIAL and df_a_cerrar.count() > 0:

    ids_con_cambio = [row["product_card_id"] for row in df_a_cerrar.select("product_card_id").collect()]

    delta_dim_producto.update(
        condition=(F.col("product_card_id").isin(ids_con_cambio)) & (F.col("es_vigente") == True),
        set={
            "es_vigente": F.lit(False),
            "fecha_fin_vigencia": F.current_date()
        }
    )

    print(f"✓ {len(ids_con_cambio)} versiones anteriores cerradas")
elif not CARGA_INICIAL:
    print("✓ No hay versiones que cerrar")

In [0]:
# ============================================================
# DIM_PRODUCTO — Celda 7: Insertar nuevas versiones (nuevos + cambiados)
# ============================================================

if not CARGA_INICIAL and df_a_insertar.count() > 0:

    max_key = spark.sql(f"SELECT MAX(producto_key) AS m FROM {GOLD_DIM_PRODUCTO_FQN}").collect()[0]["m"] or 0

    df_nuevas_versiones = (
        df_a_insertar
        .drop("producto_key_existente", "tipo_evento")
        .withColumn("producto_key", max_key + F.monotonically_increasing_id() + 1)
        .withColumn("fecha_inicio_vigencia", F.current_date())
        .withColumn("fecha_fin_vigencia", F.lit(None).cast("date"))
        .withColumn("es_vigente", F.lit(True))
    )

    df_nuevas_versiones.write.format("delta").mode("append").saveAsTable(GOLD_DIM_PRODUCTO_FQN)

    print(f"✓ {df_nuevas_versiones.count()} nuevas versiones insertadas")
elif not CARGA_INICIAL:
    print("✓ No hay nuevas versiones que insertar")

In [0]:
# ============================================================
# DIM_PRODUCTO — Celda 8: ACTUALIZACIÓN DEL LOG — EJECUCIÓN EXITOSA
# ============================================================

spark.sql(f"""
UPDATE metadata_dataco.supply_chain.hd_etl_log
SET estado = 'EXITOSO', fecha_fin = current_timestamp()
WHERE log_id = '{LOG_ID}'
""")

print("✓ etl_log actualizado: EXITOSO")